# Limpieza de Datos y Publicación en Parquet

**Objetivo:** Aplicar estrategias de limpieza basadas en los problemas detectados en el análisis de calidad y publicar los datos curados en formato Parquet particionado.

**Estrategias de limpieza:**
- **Eliminar:** Registros con errores graves (fechas futuras, duplicados)
- **Limpiar:** Normalizar valores (canales, decimales)
- **Etiquetar:** Marcar registros con problemas para análisis posterior
- **Imputar:** Rellenar valores faltantes con estrategias razonables

**Salida:** Dataset curado en formato Parquet particionado por año y mes

## 1. Configuración e importación de librerías

In [4]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import json
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from datetime import datetime
from collections import OrderedDict

from pathlib import Path


# Configurar pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print(f"Pandas version: {pd.__version__}")
print(f"PyArrow version: {pa.__version__}")
print(f"DuckDB version: {duckdb.__version__}")

Pandas version: 2.3.3
PyArrow version: 21.0.0
DuckDB version: 1.4.1


## 2. Cargar datos crudos

In [ ]:
# Definir rutas
# En notebooks, __file__ no está definido; usar el directorio de trabajo como fallback
try:
	BASE = Path(__file__).resolve().parent  # type: ignore[name-defined]
except NameError:
	BASE = Path.cwd()

DATA = BASE / "data"
RAW = DATA / "sales_raw.csv"
CURATED_DIR = DATA / "curated"

# Crear directorio para datos curados
CURATED_DIR.mkdir(exist_ok=True, parents=True)

print(f"Archivo de entrada: {RAW}")
print(f"Directorio de salida: {CURATED_DIR}")
print(f"Archivo existe: {RAW.exists()}")

NameError: name '__file__' is not defined

In [ ]:
# Leer datos crudos
df_raw = pd.read_csv(RAW, dtype=str, keep_default_na=False)

print(f"Dataset original: {df_raw.shape}")
print(f"Columnas: {list(df_raw.columns)}")
df_raw.head()

## 3. Normalización inicial de datos

In [ ]:
# Crear copia para trabajar
df = df_raw.copy()

print(f"Registros antes de limpieza: {len(df)}")

### 3.1 Limpiar espacios en blanco

In [ ]:
# Limpiar espacios en columnas de texto
for col in ["fecha", "tienda_id", "sku", "canal", "precio", "importe"]:
    df[col] = df[col].astype(str).str.strip()

print("✓ Espacios en blanco eliminados")

### 3.2 Convertir tipos de datos

In [ ]:
# UNIDADES: convertir a numérico
df["unidades"] = pd.to_numeric(df["unidades"], errors="coerce")
print(f"Unidades convertidas. Nulos: {df['unidades'].isna().sum()}")

# PRECIO: normalizar decimales (coma → punto) y convertir a numérico
df["precio"] = df["precio"].str.replace(",", ".", regex=False)
df["precio"] = pd.to_numeric(df["precio"], errors="coerce")
print(f"Precios convertidos. Nulos: {df['precio'].isna().sum()}")

# IMPORTE: normalizar decimales (coma → punto) y convertir a numérico
df["importe"] = df["importe"].str.replace(",", ".", regex=False)
df["importe"] = pd.to_numeric(df["importe"], errors="coerce")
print(f"Importes convertidos. Nulos: {df['importe'].isna().sum()}")

# FECHA: convertir a datetime
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
print(f"Fechas convertidas. Nulos: {df['fecha'].isna().sum()}")

### 3.3 Normalizar canal

In [ ]:
# Diccionario de normalización de canales
canon = {
    "web": "web",
    "tienda": "tienda",
    "app": "app",
    "web ": "web",
    " web ": "web",
    "Web": "web",
    "WEB": "web",
    "online": "web",
    "w": "web"
}

# Crear canal normalizado
df["canal_norm"] = df["canal"].str.lower().str.strip().map(canon)
df["canal_norm"] = df["canal_norm"].fillna(df["canal"].str.lower().str.strip())

print("Distribución de canal normalizado:")
print(df["canal_norm"].value_counts())
print(f"\nCanales válidos: {df['canal_norm'].isin(['tienda', 'web', 'app']).sum()}")
print(f"Canales inválidos: {(~df['canal_norm'].isin(['tienda', 'web', 'app'])).sum()}")

## 4. Etiquetar problemas de calidad (FLAGS)

Antes de eliminar o corregir datos, creamos flags para documentar los problemas encontrados.

In [ ]:
# FLAG 1: Unidades negativas
df["flag_unidades_neg"] = ((df["unidades"] < 0) & df["unidades"].notna()).astype(int)
print(f"Flag unidades negativas: {df['flag_unidades_neg'].sum()} registros")

# FLAG 2: Precio faltante o inválido
df["flag_precio_missing"] = (df["precio"].isna() | (df["precio"] <= 0)).astype(int)
print(f"Flag precio faltante/inválido: {df['flag_precio_missing'].sum()} registros")

# FLAG 3: Canal inválido
df["flag_canal_invalido"] = (~df["canal_norm"].isin(["tienda", "web", "app"])).astype(int)
print(f"Flag canal inválido: {df['flag_canal_invalido'].sum()} registros")

# FLAG 4: Fecha inválida (nula o futura)
df["flag_fecha_invalida"] = (df["fecha"].isna() | (df["fecha"] > pd.Timestamp.today())).astype(int)
print(f"Flag fecha inválida: {df['flag_fecha_invalida'].sum()} registros")

# FLAG 5: Importe inconsistente (lo calculamos después de limpiar precios)
df["flag_importe_inconsistente"] = 0

In [ ]:
# Resumen de flags
print("\n" + "="*60)
print("RESUMEN DE FLAGS (PROBLEMAS DETECTADOS)")
print("="*60)
print(f"Unidades negativas:       {df['flag_unidades_neg'].sum():6d} ({df['flag_unidades_neg'].mean()*100:5.2f}%)")
print(f"Precio faltante/inválido: {df['flag_precio_missing'].sum():6d} ({df['flag_precio_missing'].mean()*100:5.2f}%)")
print(f"Canal inválido:           {df['flag_canal_invalido'].sum():6d} ({df['flag_canal_invalido'].mean()*100:5.2f}%)")
print(f"Fecha inválida:           {df['flag_fecha_invalida'].sum():6d} ({df['flag_fecha_invalida'].mean()*100:5.2f}%)")
print("="*60)

## 5. Estrategias de limpieza

### 5.1 LIMPIAR: Normalizar canal

In [ ]:
# Reemplazar canal original por el normalizado
df["canal"] = df["canal_norm"]
df = df.drop(columns=["canal_norm"])

print("✓ Canal normalizado aplicado")
print("\nDistribución final de canales:")
print(df["canal"].value_counts())

### 5.2 IMPUTAR: Precio faltante

In [ ]:
# Estrategia: imputar precio faltante con la mediana por SKU
# Si no hay datos del SKU, usar mediana global

print(f"Precios nulos antes de imputación: {df['precio'].isna().sum()}")

# Calcular mediana por SKU
median_price_per_sku = df.groupby("sku")["precio"].median()

# Imputar usando mediana del SKU
def impute_price(row):
    if pd.isna(row["precio"]):
        sku_median = median_price_per_sku.get(row["sku"], np.nan)
        return sku_median if pd.notna(sku_median) else df["precio"].median()
    return row["precio"]

df["precio"] = df.apply(impute_price, axis=1)

# Si aún quedan nulos, usar mediana global
if df["precio"].isna().any():
    global_median = df["precio"].median()
    df["precio"] = df["precio"].fillna(global_median)
    print(f"Mediana global usada: {global_median:.2f}")

print(f"Precios nulos después de imputación: {df['precio'].isna().sum()}")
print(f"✓ Precios imputados")

### 5.3 RECOMPUTAR: Importe faltante

In [ ]:
# Si el importe está ausente pero tenemos unidades y precio, recalcularlo
print(f"Importes nulos antes de recálculo: {df['importe'].isna().sum()}")

mask = df["importe"].isna() & df["unidades"].notna() & df["precio"].notna()
df.loc[mask, "importe"] = df.loc[mask, "unidades"] * df.loc[mask, "precio"]

print(f"Importes recalculados: {mask.sum()}")
print(f"Importes nulos después de recálculo: {df['importe'].isna().sum()}")
print("✓ Importes recalculados donde era posible")

### 5.4 ETIQUETAR: Importe inconsistente

In [ ]:
# Marcar registros donde importe ≠ unidades × precio (tolerancia ±2%)
test_df = df.dropna(subset=["unidades", "precio", "importe"]).copy()

if len(test_df) > 0:
    expected_importe = test_df["unidades"] * test_df["precio"]
    actual_importe = test_df["importe"]
    diff = (actual_importe - expected_importe).abs()
    tolerance = 0.02 * (expected_importe.abs() + 1e-9)
    
    inconsistent = diff > tolerance
    df.loc[test_df.index[inconsistent], "flag_importe_inconsistente"] = 1
    
    print(f"Registros con importe inconsistente: {inconsistent.sum()}")
    print(f"Porcentaje: {(inconsistent.sum()/len(test_df)*100):.2f}%")
    
    if inconsistent.sum() > 0:
        print("\nEjemplos de inconsistencias:")
        incon_df = test_df[inconsistent].copy()
        incon_df["expected"] = expected_importe[inconsistent]
        incon_df["diff"] = diff[inconsistent]
        display(incon_df[["fecha", "sku", "unidades", "precio", "importe", "expected", "diff"]].head(5))

### 5.5 ELIMINAR: Fechas inválidas

In [ ]:
# Eliminar registros con fecha nula o futura
print(f"Registros antes de eliminar fechas inválidas: {len(df)}")

df = df[df["fecha"].notna()]
print(f"Registros después de eliminar fechas nulas: {len(df)}")

df = df[df["fecha"] <= pd.Timestamp.today()]
print(f"Registros después de eliminar fechas futuras: {len(df)}")

print("✓ Fechas inválidas eliminadas")

### 5.6 ELIMINAR: Duplicados

In [ ]:
# Eliminar duplicados por clave (fecha, tienda_id, sku, canal)
# Mantener el primer registro
print(f"Registros antes de deduplicar: {len(df)}")

# Ordenar primero para mantener un orden consistente
df = df.sort_values(["fecha", "tienda_id", "sku", "canal"])

# Identificar duplicados antes de eliminar
duplicates_mask = df.duplicated(subset=["fecha", "tienda_id", "sku", "canal"], keep="first")
n_duplicates = duplicates_mask.sum()

# Eliminar duplicados
df = df.drop_duplicates(subset=["fecha", "tienda_id", "sku", "canal"], keep="first")

print(f"Duplicados eliminados: {n_duplicates}")
print(f"Registros después de deduplicar: {len(df)}")
print("✓ Duplicados eliminados")

## 6. Preparar datos para exportación

In [ ]:
# Agregar columnas de partición (año y mes)
df["anio"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month

# Convertir fecha a date (sin hora)
df["fecha"] = df["fecha"].dt.date

print("Columnas de partición agregadas:")
print(f"  - anio: {df['anio'].min()} a {df['anio'].max()}")
print(f"  - mes: {df['mes'].min()} a {df['mes'].max()}")

print("\nDistribución por año-mes:")
print(df.groupby(["anio", "mes"]).size().head(20))

In [ ]:
# Reordenar columnas para mejor legibilidad
columns_order = [
    "fecha", "anio", "mes",
    "tienda_id", "sku", "canal",
    "unidades", "precio", "importe",
    "flag_unidades_neg", "flag_precio_missing", "flag_canal_invalido",
    "flag_fecha_invalida", "flag_importe_inconsistente"
]

df = df[columns_order]

print("Dataset limpio final:")
print(f"Shape: {df.shape}")
print(f"\nPrimeras filas:")
df.head()

In [ ]:
# Verificar tipos de datos
print("Tipos de datos del dataset limpio:")
df.info()

## 7. Exportar a Parquet particionado

In [ ]:
# Limpiar directorio de salida si existe
import shutil

if CURATED_DIR.exists() and any(CURATED_DIR.iterdir()):
    print(f"Limpiando directorio {CURATED_DIR}...")
    shutil.rmtree(CURATED_DIR)
    CURATED_DIR.mkdir(exist_ok=True, parents=True)

print(f"Directorio de salida preparado: {CURATED_DIR}")

In [ ]:
# Convertir a PyArrow Table
table = pa.Table.from_pandas(df)

print("Esquema de la tabla PyArrow:")
print(table.schema)

In [ ]:
# Escribir dataset particionado
print("Escribiendo dataset particionado por anio y mes...")

pq.write_to_dataset(
    table,
    root_path=str(CURATED_DIR),
    partition_cols=["anio", "mes"],
    compression="snappy",
    existing_data_behavior="overwrite_or_ignore"
)

print("✓ Dataset exportado exitosamente")

## 8. Verificar estructura de archivos Parquet

In [ ]:
# Listar estructura de directorios
print("Estructura de archivos generados:\n")

def print_tree(directory, prefix="", max_depth=3, current_depth=0):
    if current_depth >= max_depth:
        return
    
    items = sorted(directory.iterdir(), key=lambda x: (not x.is_dir(), x.name))
    for i, item in enumerate(items):
        is_last = i == len(items) - 1
        current_prefix = "└── " if is_last else "├── "
        print(f"{prefix}{current_prefix}{item.name}")
        
        if item.is_dir():
            extension_prefix = "    " if is_last else "│   "
            print_tree(item, prefix + extension_prefix, max_depth, current_depth + 1)

print(f"{CURATED_DIR.name}/")
print_tree(CURATED_DIR)

In [ ]:
# Calcular tamaños de archivos
parquet_files = list(CURATED_DIR.rglob("*.parquet"))
total_parquet_size = sum(f.stat().st_size for f in parquet_files)
csv_size = RAW.stat().st_size

print("\n" + "="*60)
print("COMPARACIÓN DE TAMAÑOS")
print("="*60)
print(f"CSV original:        {csv_size:>12,} bytes ({csv_size/1024:.2f} KB)")
print(f"Parquet total:       {total_parquet_size:>12,} bytes ({total_parquet_size/1024:.2f} KB)")
print(f"Reducción:           {(1 - total_parquet_size/csv_size)*100:>11.2f}%")
print(f"Factor compresión:   {csv_size/total_parquet_size:>15.2f}x")
print(f"\nArchivos Parquet:    {len(parquet_files):>12,}")
print(f"Tamaño promedio:     {total_parquet_size/len(parquet_files):>12,.0f} bytes")
print("="*60)

## 9. Calcular métricas de calidad DESPUÉS de limpieza

In [ ]:
def compute_quality_metrics(df, total):
    """Calcula todas las métricas de calidad"""
    def ratio(x):
        return round(float(x) / total, 4) if total > 0 else 0.0
    
    metrics = OrderedDict()
    
    # 1. Completitud
    metrics["completeness"] = {
        "precio": {"value": ratio(df["precio"].notna().sum()), "threshold": 0.99},
        "unidades": {"value": ratio(df["unidades"].notna().sum()), "threshold": 0.99},
        "importe": {"value": ratio(df["importe"].notna().sum()), "threshold": 0.99},
    }
    
    # 2. Unicidad
    key = df[["fecha", "tienda_id", "sku"]].astype(str).agg("|".join, axis=1)
    dup_rate = key.duplicated(keep=False).mean()
    metrics["uniqueness"] = {
        "fecha_tienda_sku": {"value": round(1.0 - float(dup_rate), 4), "threshold": 1.0}
    }
    
    # 3. Validez
    valid_canal = df["canal"].isin(["tienda", "web", "app"]).mean()
    valid_precio = ((df["precio"] > 0) & df["precio"].notna()).mean()
    valid_unidades = ((df["unidades"] >= 0) & df["unidades"].notna()).mean()
    valid_fecha = (df["fecha"].notna()).mean()
    
    metrics["validity"] = {
        "canal_in_set": {"value": round(float(valid_canal), 4), "threshold": 1.0},
        "precio_gt_0": {"value": round(float(valid_precio), 4), "threshold": 0.99},
        "unidades_ge_0": {"value": round(float(valid_unidades), 4), "threshold": 0.99},
        "fecha_valida": {"value": round(float(valid_fecha), 4), "threshold": 1.0},
    }
    
    # 4. Consistencia
    test = df.dropna(subset=["unidades", "precio", "importe"])
    if len(test) > 0:
        expected = test["unidades"] * test["precio"]
        diff = (test["importe"] - expected).abs()
        tolerance = 0.02 * (expected.abs() + 1e-9)
        consistent = (diff <= tolerance).mean()
    else:
        consistent = 0.0
    
    metrics["consistency"] = {
        "importe_eq_unidades_x_precio_tol2pct": {"value": round(float(consistent), 4), "threshold": 0.98}
    }
    
    # 5. Puntualidad
    max_date = pd.to_datetime(df["fecha"]).max()
    lag_days = (pd.Timestamp.today().normalize() - max_date).days if pd.notna(max_date) else None
    metrics["timeliness"] = {
        "max_date": str(max_date.date() if pd.notna(max_date) else None),
        "lag_days": int(lag_days) if lag_days is not None else None,
        "sla": "D+1 10:00"
    }
    
    return metrics

total_rows_after = len(df)
metrics_after = compute_quality_metrics(df, total_rows_after)

print("✓ Métricas de calidad calculadas")

In [ ]:
# Mostrar métricas DESPUÉS de limpieza
print("=" * 60)
print("MÉTRICAS DE CALIDAD (DESPUÉS DE LIMPIEZA)")
print("=" * 60)
print(f"\nTotal de registros: {total_rows_after}\n")

all_checks_after = []

for dimension, metrics in metrics_after.items():
    if dimension == "timeliness":
        print(f"\n{dimension.upper()}:")
        print(f"  Fecha máxima: {metrics['max_date']}")
        print(f"  Días de retraso: {metrics['lag_days']}")
        print(f"  SLA: {metrics['sla']}")
        continue
    
    print(f"\n{dimension.upper()}:")
    for metric, data in metrics.items():
        if isinstance(data, dict) and "value" in data:
            value_pct = data["value"] * 100
            threshold_pct = data["threshold"] * 100
            passed = data["value"] >= data["threshold"]
            all_checks_after.append(passed)
            status = "✓" if passed else "✗"
            print(f"  {status} {metric:40s} | {value_pct:6.2f}% (umbral: {threshold_pct:6.2f}%)")

print(f"\n{'-'*60}")
checks_passed_after = sum(all_checks_after)
total_checks_after = len(all_checks_after)
quality_score_after = (checks_passed_after / total_checks_after * 100) if total_checks_after > 0 else 0
print(f"PUNTUACIÓN DE CALIDAD: {quality_score_after:.1f}% ({checks_passed_after}/{total_checks_after} checks pasados)")
print("=" * 60)

## 10. Generar reporte de calidad completo

In [ ]:
# Cargar métricas ANTES (si existen)
metrics_before_file = BASE / "quality_metrics_before.json"

if metrics_before_file.exists():
    with open(metrics_before_file, "r", encoding="utf-8") as f:
        report_before = json.load(f)
        metrics_before = report_before.get("metrics", {})
        rows_before = report_before.get("summary", {}).get("total_rows", 0)
    print(f"✓ Métricas ANTES cargadas desde {metrics_before_file}")
else:
    metrics_before = {}
    rows_before = len(df_raw)
    print("⚠ No se encontraron métricas ANTES. Usando valores por defecto.")

In [ ]:
# Crear reporte completo
quality_report = {
    "dataset": "sales",
    "version": datetime.now().strftime("%Y_%m_%d"),
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "summary": {
        "rows_before": int(rows_before),
        "rows_after": int(total_rows_after),
        "rows_removed": int(rows_before - total_rows_after),
        "removal_rate_pct": round((rows_before - total_rows_after) / rows_before * 100, 2) if rows_before > 0 else 0,
        "cols": int(df.shape[1]),
        "file_sizes_bytes": {
            "csv": int(csv_size),
            "parquet": int(total_parquet_size)
        },
        "compression_ratio": round(csv_size / total_parquet_size, 2) if total_parquet_size > 0 else 0,
        "partitions": {}
    },
    "metrics": {
        "before": metrics_before,
        "after": metrics_after
    },
    "quality_score": {
        "after": round(quality_score_after, 2)
    },
    "cleaning_actions": [
        "Normalización de canal (web, tienda, app)",
        "Conversión de decimales (coma → punto)",
        "Imputación de precios faltantes (mediana por SKU)",
        "Recálculo de importes faltantes",
        "Eliminación de fechas inválidas (nulas o futuras)",
        "Eliminación de duplicados",
        "Flags de calidad agregados"
    ],
    "notes": "Este informe resume métricas antes/después de limpieza y decisiones aplicadas."
}

# Información de particiones
parts = {}
for anio_dir in sorted(CURATED_DIR.glob("anio=*")):
    anio = anio_dir.name.split("=")[1]
    months = sorted([mes_dir.name.split("=")[1] for mes_dir in anio_dir.glob("mes=*")])
    parts[anio] = months
quality_report["summary"]["partitions"] = parts

print("✓ Reporte de calidad generado")

In [ ]:
# Guardar reporte
report_file = BASE / "quality_report.json"
with open(report_file, "w", encoding="utf-8") as f:
    json.dump(quality_report, f, indent=2, ensure_ascii=False)

print(f"✓ Reporte guardado en: {report_file}")
print(f"\nContenido del reporte:")
print(json.dumps(quality_report, indent=2, ensure_ascii=False)[:2000] + "\n...")

## 11. Generar SQL para verificación con DuckDB

In [ ]:
# Leer plantilla SQL
template_sql_file = BASE / "duckdb_checks_template.sql"

if template_sql_file.exists():
    template_sql = template_sql_file.read_text(encoding="utf-8")
    print(f"✓ Plantilla SQL cargada desde {template_sql_file}")
else:
    print("⚠ No se encontró la plantilla SQL")
    template_sql = ""

In [ ]:
# Crear SQL de verificación
curated_path = str(CURATED_DIR).replace("\\", "/")

checks_sql = f"""-- Checks sobre dataset Parquet particionado
-- Generado automáticamente por limpieza_y_parquet.ipynb
-- Fecha: {datetime.now().isoformat()}

INSTALL parquet;
LOAD parquet;

-- Crear vista sobre los datos curados
CREATE OR REPLACE VIEW sales_curated AS
SELECT * FROM read_parquet('{curated_path}/anio=*/mes=*/*.parquet');

-- Información general
SELECT 'Total de registros' AS descripcion, COUNT(*) AS valor FROM sales_curated;

SELECT 'Rango de fechas' AS descripcion, 
       MIN(fecha) || ' a ' || MAX(fecha) AS valor 
FROM sales_curated;

"""

# Reemplazar NOMBRE_TABLA en la plantilla
if template_sql:
    template_checks = template_sql.replace("NOMBRE_TABLA", "sales_curated")
    # Eliminar comentarios sobre .mode y .timer (no soportados en Python)
    template_checks = "\n".join([line for line in template_checks.split("\n") 
                                  if not line.strip().startswith("-- .")])
    checks_sql += "\n" + template_checks

# Guardar SQL
sql_file = BASE / "duckdb_checks.sql"
sql_file.write_text(checks_sql, encoding="utf-8")

print(f"✓ SQL de verificación guardado en: {sql_file}")
print(f"\nPrimeras líneas del SQL generado:")
print(checks_sql[:800] + "\n...")

## 12. Verificar datos con DuckDB

In [ ]:
# Ejecutar verificaciones con DuckDB
print("Ejecutando verificaciones con DuckDB...\n")

con = duckdb.connect()

try:
    # Ejecutar el SQL
    con.execute(checks_sql)
    
    # Obtener resultados de las consultas
    print("✓ SQL ejecutado exitosamente")
    print("\nLos checks de calidad se encuentran en el archivo duckdb_checks.sql")
    print("Puedes ejecutarlos con: duckdb -c '.read duckdb_checks.sql'")
    
except Exception as e:
    print(f"⚠ Error al ejecutar SQL: {e}")
finally:
    con.close()

## 13. Resumen final

In [ ]:
print("\n" + "="*70)
print("RESUMEN DE LIMPIEZA Y PUBLICACIÓN")
print("="*70)

print(f"\n📊 DATOS:")
print(f"  Registros originales:     {rows_before:>10,}")
print(f"  Registros finales:        {total_rows_after:>10,}")
print(f"  Registros eliminados:     {rows_before - total_rows_after:>10,} ({(rows_before - total_rows_after)/rows_before*100:5.2f}%)")

print(f"\n📁 ARCHIVOS:")
print(f"  CSV original:             {csv_size:>10,} bytes ({csv_size/1024:8.2f} KB)")
print(f"  Parquet total:            {total_parquet_size:>10,} bytes ({total_parquet_size/1024:8.2f} KB)")
print(f"  Reducción de tamaño:      {(1 - total_parquet_size/csv_size)*100:>10.2f}%")
print(f"  Factor de compresión:     {csv_size/total_parquet_size:>14.2f}x")
print(f"  Archivos Parquet:         {len(parquet_files):>10,}")

print(f"\n✅ CALIDAD:")
print(f"  Puntuación final:         {quality_score_after:>13.1f}%")
print(f"  Checks pasados:           {checks_passed_after:>10,} / {total_checks_after}")

print(f"\n📂 PARTICIONES:")
for anio, meses in parts.items():
    print(f"  Año {anio}:                {', '.join(f'mes {m}' for m in meses)}")

print(f"\n📝 ARCHIVOS GENERADOS:")
print(f"  • {CURATED_DIR.relative_to(BASE)}/          (datos Parquet particionados)")
print(f"  • quality_report.json        (reporte de calidad)")
print(f"  • duckdb_checks.sql          (verificaciones DuckDB)")

print("\n" + "="*70)
print("✓ LIMPIEZA Y PUBLICACIÓN COMPLETADA")
print("="*70)

## 14. Próximos pasos

### Para verificar los datos con DuckDB CLI:
```bash
duckdb -c ".read duckdb_checks.sql"
```

### Para el informe de calidad:
1. Abre la plantilla `UD1_Informe_Calidad_template.md`
2. Completa con los datos de `quality_report.json`
3. Documenta:
   - Decisiones de limpieza tomadas
   - Impacto de CSV vs Parquet
   - Métricas antes → después
   - Ejemplos de problemas corregidos

### Para análisis adicional:
- Usa DuckDB para consultar el dataset particionado
- Analiza los flags de calidad
- Mide tiempos de consulta vs CSV original